# Reproduce the CemCT manuscript analyses

This notebook analyses the supplied experimental 200³ labelled volume through all six CemCT modules. It uses the installed package API and relative data paths. Results are written to `results/manuscript/<run timestamp>/`, leaving source data unchanged.

## Setup and final reproduction configuration
From the extracted project root, activate your CemCT Python 3.11 environment and install `python -m pip install -e ".[all]"`. Install `jupyterlab ipykernel` if needed, and select that environment as the notebook kernel. Open this notebook and run cells in order. Scientific calculations on 200³ voxels may take several minutes and use substantial memory.

The final configuration uses 0.7 um voxels, label 2, 3000 walkers, 5000 steps, stride 10 and the union of through-connected masks. Random-walk MSDs are fitted through the origin over **all 500 saved time points: steps 0–4990 inclusive** (`fit_start_fraction=0.0`, `fit_end_fraction=1.0`). `deterministic_seed=True` enables PyTrax's Boolean debugging seed; it is not an integer seed. One process is used.

The full-interval fit follows the convention present in the older notebook and reproduces the manuscript's rounded random-walk values with this experimental volume. Parameter provenance is recorded in the JSON file. The final comparison cell reports numerical differences from rounded manuscript values; it is not an uncertainty assessment. The original teaching notebooks are retained with their separate example settings.


In [ ]:
from pathlib import Path
import json, hashlib, platform
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError

def find_root():
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "pyproject.toml").is_file() and (p / "data/experimental/manifest.json").is_file():
            return p
    raise FileNotFoundError("Start Jupyter from the extracted CemCT-main folder or its examples folder.")
ROOT = find_root()
CONFIG = json.loads((ROOT / "examples/manuscript_parameters.json").read_text())
MANIFEST = json.loads((ROOT / "data/experimental/manifest.json").read_text())
DATA = ROOT / "data/experimental"
INPUT = DATA / "NasSO4_slag_raw_BC_Seg.tif"
for name, expected in MANIFEST["sha256"].items():
    assert hashlib.sha256((DATA / name).read_bytes()).hexdigest() == expected, name
print(json.dumps(CONFIG, indent=2))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
from IPython.display import display
import cemct
from cemct.io import load_labelled_tiff
from cemct.phase_fraction import calculate_phase_fractions
from cemct.connectivity import analyse_connectivity
from cemct.pore_size import calculate_pore_size_distribution
from cemct.diffusion import calculate_directional_diffusion
from cemct.random_walk import calculate_connected_random_walk
from cemct.permeability import analyse_permeability
from cemct.export import export_analysis_bundle

print("CemCT imported from:", cemct.__file__)
OUT = ROOT / "results/manuscript" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUT.mkdir(parents=True)
versions = {}
for name in ["cemct", "numpy", "scipy", "pandas", "tifffile", "porespy", "openpnm", "pyamg", "pytrax", "matplotlib"]:
    try: versions[name] = version(name)
    except PackageNotFoundError: versions[name] = "unavailable"
META = {"configuration": CONFIG, "input_manifest": MANIFEST, "versions": versions,
        "python": platform.python_version(), "platform": platform.platform(),
        "array_order": "ZYX", "cemct_import_path": str(cemct.__file__)}
(OUT / "run_metadata.json").write_text(json.dumps(META, indent=2))
def save(name, **kwargs):
    return export_analysis_bundle(input_file=INPUT, analysis_name=name,
                                  output_directory=OUT / name, metadata=META, **kwargs)
volume = load_labelled_tiff(INPUT)
assert tuple(volume.shape) == tuple(MANIFEST["shape_zyx"])
labels, counts = np.unique(volume, return_counts=True)
assert dict(zip(map(str, labels), map(int, counts))) == MANIFEST["label_counts"]
for name in MANIFEST["sha256"]:
    with tifffile.TiffFile(DATA / name) as t:
        assert np.isclose(t.imagej_metadata["spacing"], CONFIG["voxel_size_um"])
        for axis in ["XResolution", "YResolution"]:
            n, d = t.pages[0].tags[axis].value
            assert np.isclose(d / n, CONFIG["voxel_size_um"])
print("Input counts and calibration verified. Outputs:", OUT)


## Phase fractions and central slices


In [ ]:
phases = calculate_phase_fractions(volume, {int(k): v for k, v in CONFIG["phases"].items()}, CONFIG["voxel_size_um"])
display(phases)
raw = tifffile.imread(DATA / "raw_CT.tif")
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
z = volume.shape[0] // 2
axes[0].imshow(raw[z], cmap="gray"); axes[0].set_title("Grayscale central Z slice")
axes[1].imshow(volume[z], cmap="viridis", vmin=0, vmax=2); axes[1].set_title("Labels 0 / 1 / 2")
axes[2].bar(phases["label"].astype(str), phases["volume_fraction_percent"])
axes[2].set(xlabel="Label", ylabel="Volume (%)", title="Phase fractions")
fig.tight_layout()
save("phase_fraction", tables={"Phase fractions": phases}, figure=fig)
plt.show()
del raw


## Strict face connectivity


In [ ]:
connectivity = analyse_connectivity(volume, phase_label=CONFIG["phase_label"])
display(pd.DataFrame([connectivity["summary"]]))
save("connectivity", tables={"Summary": pd.DataFrame([connectivity["summary"]])}, masks=connectivity["masks"])


## Local thickness pore size


In [ ]:
pore_size = calculate_pore_size_distribution(volume, phase_label=CONFIG["phase_label"], voxel_size_um=CONFIG["voxel_size_um"], **CONFIG["pore_size"])
display(pd.DataFrame([pore_size["summary"]]))
save("pore_size", tables={"Summary": pd.DataFrame([pore_size["summary"]]), "Distribution": pore_size["distribution"], "Cumulative": pore_size["cumulative"], "Size classes": pore_size["size_classes"]}, scalar_volumes={"local_diameter_um": pore_size["local_diameter_um"]})


## Finite difference diffusion


In [ ]:
diffusion = calculate_directional_diffusion(volume, phase_label=CONFIG["phase_label"], **CONFIG["diffusion"])
display(diffusion["summary"])
save("diffusion", tables={"Summary": diffusion["summary"]}, scalar_volumes=diffusion["concentration_fields"])


## Random walk transport

Fit all saved points (steps 0–4990 inclusive) through the origin. The fit interval is 0.0–1.0. This uses the fixed PyTrax debugging-seed option and does not establish uncertainty from independent random seeds.


In [ ]:
random_walk = calculate_connected_random_walk(volume, phase_label=CONFIG["phase_label"], **CONFIG["random_walk"])
display(pd.DataFrame([random_walk["summary"]]))
save("random_walk", tables={"Summary": pd.DataFrame([random_walk["summary"]]), "MSD": random_walk["msd"], "Fits": pd.DataFrame(random_walk["fit_results"]).T}, masks={"transport_mask": random_walk["transport_mask"]})


## Pore network permeability
Pressure outputs are network node arrays in NPZ, not three-dimensional voxel pressure fields.


In [ ]:
permeability = analyse_permeability(volume, phase_label=CONFIG["phase_label"], voxel_size_um=CONFIG["voxel_size_um"], **CONFIG["permeability"])
display(permeability["summary"])
arrays = {}
for direction, network in permeability["networks"].items():
    arrays[direction + "_pore_coordinates"] = np.asarray(network["pore.coords"])
    arrays[direction + "_throat_connections"] = np.asarray(network["throat.conns"])
    arrays[direction + "_pore_pressure"] = permeability["pressure_fields"][direction]
save("permeability", tables={"Summary": permeability["summary"]}, arrays=arrays)


## Compare against the manuscript
Reported values below are rounded manuscript values, not newly generated reference results. Differences are displayed without imposing invented acceptance tolerances. Confirm remaining parameters and investigate differences before publication.


In [ ]:
rows = []
def compare(metric, actual, reported, unit):
    rows.append({"metric": metric, "calculated": float(actual), "manuscript_rounded": reported, "difference": float(actual)-reported, "unit": unit})
for label, reported in [(0,29.0),(1,53.5),(2,17.5)]:
    compare(f"Phase {label}", phases.set_index("label").loc[label,"volume_fraction_percent"], reported, "%")
cs = connectivity["summary"]
compare("Boundary accessible pores", 100*cs["boundary_accessible_fraction"],98.3,"% of pores")
compare("Isolated pores",100*cs["isolated_fraction"],1.7,"% of pores")
for d in "XYZ": compare(d+" connectivity",100*cs[d+"_connectivity"],95.5,"% of pores")
for key, val in [("minimum_local_diameter_um",1.4),("median_local_diameter_um",3.4),("maximum_local_diameter_um",7.1)]: compare(key,pore_size["summary"][key],val,"um")
fd = diffusion["summary"].set_index("direction")
pn = permeability["summary"].set_index("Direction")
for d,F,De,tfd,trw,k in zip("XYZ",[62.0,69.9,58.2],[0.016,0.014,0.017],[10.4,11.7,9.8],[5.0,5.7,4.6],[5.1e-15,4.7e-15,5.0e-15]):
    compare(d+" formation factor",fd.loc[d,"formation_factor"],F,"dimensionless")
    compare(d+" relative diffusivity",fd.loc[d,"relative_effective_diffusivity"],De,"dimensionless")
    compare(d+" diffusion tortuosity",fd.loc[d,"diffusion_tortuosity"],tfd,"dimensionless")
    compare(d+" random-walk tortuosity",random_walk["summary"][d+"_tortuosity"],trw,"dimensionless")
    compare(d+" permeability",pn.loc[d,"Intrinsic permeability (m^2)"],k,"m^2")
comparison = pd.DataFrame(rows)
comparison.to_csv(OUT / "manuscript_comparison.csv", index=False)
display(comparison)
print("All six analysis stages completed. Review differences in", OUT)
